# ROGII — Kaggle inference (kernels-only submission)

This competition accepts **notebook** submissions only. This notebook runs on Kaggle with
**internet OFF**, so it loads everything from attached **Datasets**:

1. **`rogii-code`** — this repo's `src/` folder (upload it as a dataset).
2. **`rogii-models`** — trained artifacts `probs/<version>/{oof,test}.npy` (+ any saved models),
   produced on Colab and uploaded as a dataset.
3. The competition data is auto-mounted at `/kaggle/input/rogii-wellbore-geology-prediction/` (src.config detects it).

Output: `/kaggle/working/submission.csv`. Then **Submit** this notebook to the competition.
See `docs/submission_workflow.md` for the full handoff.

## 1. Put attached code on the path + point at the data

In [ ]:
import sys, os
# Attached code dataset (this repo's src/). Adjust the path to your dataset slug.
sys.path.insert(0, "/kaggle/input/rogii-code")          # so `import src...` works
os.environ.setdefault("ROGII_DATA_DIR", "/kaggle/input/rogii-wellbore-geology-prediction")  # explicit; config also auto-detects

from src import data, features, submission           # noqa: E402
import numpy as np                                    # noqa: E402
print("test wells:", data.list_well_ids("test"))


## 2. Build test features (post-PS rows = the submission rows)

In [ ]:
ds = features.build_dataset("test", with_alignment=True, post_ps_only=True)
X_test, groups = ds["X"], ds["groups"]
print("X_test:", X_test.shape)


## 3. Load model artifact(s) + predict

Single model: load its saved booster and `.predict(X_test)`. Blend: load each member's test preds and combine with the OOF-fitted weights (see src.blend). Below is the single-model shape — adapt to your trained artifact.

In [ ]:
import joblib, glob
MODEL_DIR = "/kaggle/input/rogii-models"

# EXAMPLE (single LightGBM saved with joblib.dump(model, "model.pkl") on Colab):
model_paths = sorted(glob.glob(f"{MODEL_DIR}/**/*.pkl", recursive=True))
assert model_paths, f"no model artifacts found under {MODEL_DIR} — attach the rogii-models dataset"
model = joblib.load(model_paths[0])
test_pred = model.predict(X_test)

# (Blend variant: load several members' test preds + use src.blend.apply_blend with OOF weights.)
print("predictions:", test_pred.shape, "range", float(test_pred.min()), float(test_pred.max()))


## 4. Assemble per-well predictions → submission.csv

In [ ]:
# Map flat post-PS predictions back to per-well arrays (groups gives the well per row).
well_preds = {}
for wid in dict.fromkeys(groups):           # preserves order
    well_preds[wid] = test_pred[groups == wid]

ss = submission.build_submission_from_wells(well_preds, split="test")
ss.to_csv("/kaggle/working/submission.csv", index=False)
print("wrote /kaggle/working/submission.csv", ss.shape)
ss.head()
